In [1]:
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict
from sklearn.cluster import HDBSCAN
import xgboost as xgb
from tqdm import tqdm
import joblib


class DateRangeScaler:
    def __init__(self):
        self.means = None
        self.stds = None
        self.fitted = False

    def fit(self, df: pd.DataFrame, feature_cols: List[str]):
        self.means = df[feature_cols].mean()
        self.stds = df[feature_cols].std().replace(0, 1)
        self.fitted = True

    def transform(self, df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        if not self.fitted:
            raise ValueError("Scaler not fitted")

        df_scaled = df.copy()
        df_scaled[feature_cols] = (df[feature_cols] - self.means) / self.stds
        return df_scaled

    def fit_transform(self, df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        self.fit(df, feature_cols)
        return self.transform(df, feature_cols)


class DataManager:
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: List[str],
        delay_cols: List[str] = ["1_w_delay", "1_m_delay", "target_return"],
        date_col: str = "TRADEDATE",
        id_col: str = "SECID",
        use_scaler: bool = False,
        horizon_days: int = 180,
    ):  
        self.df = df.copy()
        self.secid2id = {s: i for i, s in enumerate(sorted(self.df["SECID"].unique()))}
        self.id2secid = {i: s for s, i in self.secid2id.items()}
        self.df["SECID_raw"] = self.df["SECID"]
        self.df["SECID"] = self.df["SECID"].map(self.secid2id)
        self.feature_cols = feature_cols
        self.delay_cols = delay_cols
        self.date_col = date_col
        self.id_col = id_col
        self.horizon_days = horizon_days

        self.use_scaler = use_scaler
        self.scaler = DateRangeScaler() if use_scaler else None

        self.df[self.date_col] = pd.to_datetime(self.df[self.date_col])
        self.df = self.df.sort_values([self.date_col, self.id_col]).reset_index(drop=True)

        self.available_dates = sorted(self.df[self.date_col].unique())

    def fit_scaler(self, start_date: str, end_date: str):
        """
        Fit scaler on data within specified date range
        """
        if not self.use_scaler:
            return

        mask = (self.df[self.date_col] >= start_date) & (self.df[self.date_col] <= end_date)
        df_slice = self.df.loc[mask]

        self.scaler.fit(df_slice, self.feature_cols)

    def apply_scaler(self):
        """
        Apply fitted scaler to entire dataset
        """
        if not self.use_scaler:
            return

        self.df = self.scaler.transform(self.df, self.feature_cols)

    def _get_nearest_available_date(self, target_date: pd.Timestamp) -> pd.Timestamp:
        """
        Returns the nearest available date <= target_date
        """
        valid_dates = [d for d in self.available_dates if d <= target_date]
        if not valid_dates:
            return None
        return max(valid_dates)

    def get_snapshot_with_lag(
        self,
        reference_date: str,
        lag_days: int,
    ) -> pd.DataFrame:
        """
        Returns data for reference_date minus lag_days using nearest available date,
        accounting for horizon to avoid lookahead bias
        """
        reference_date = pd.to_datetime(reference_date)
        target_date = reference_date - pd.Timedelta(days=lag_days)

        nearest_date = self._get_nearest_available_date(target_date)
        if nearest_date is None:
            return pd.DataFrame()

        max_date = self.df[self.date_col].max()
        cutoff_date = max_date - pd.Timedelta(days=self.horizon_days)

        if nearest_date > cutoff_date:
            return pd.DataFrame()

        df_slice = self.df[self.df[self.date_col] == nearest_date]

        cols = [self.id_col, self.date_col] + self.feature_cols + self.delay_cols
        return df_slice[cols].dropna()

    def get_multi_horizon_snapshots(
        self,
        reference_date: str,
        lags: List[int] = [14, 42, 182],
    ) -> Dict[int, pd.DataFrame]:
        """
        Returns data snapshots for multiple lag horizons
        """
        result = {}

        for lag in lags:
            result[lag] = self.get_snapshot_with_lag(reference_date, lag)

        return result

    def get_training_data(
        self,
        start_date: str,
        end_date: str,
        target_col: str = "target_return",
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Returns training data filtered by date range and horizon cutoff
        """
        max_date = self.df[self.date_col].max()
        cutoff_date = max_date - pd.Timedelta(days=self.horizon_days)

        mask = (
            (self.df[self.date_col] >= start_date) &
            (self.df[self.date_col] <= end_date) &
            (self.df[self.date_col] <= cutoff_date)
        )

        df_slice = self.df.loc[mask]
        df_slice = df_slice.dropna(subset=self.feature_cols + [target_col])

        X = df_slice[self.feature_cols].values
        y = df_slice[target_col].values

        return X, y

    def get_available_dates(self) -> List[pd.Timestamp]:
        return self.available_dates

    def get_features_by_date(self, date: str) -> pd.DataFrame:
        date = pd.to_datetime(date)
        return self.df[self.df[self.date_col] == date][
            [self.id_col] + self.feature_cols
        ].copy()

    def decode_secid(self, secid_encoded):
        return self.id2secid.get(secid_encoded, secid_encoded)


class XGBModelWrapper:
    def __init__(self):
        self.model = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42,
        )

    def fit(self, X, y):
        if len(X) == 0:
            return
        self.model.fit(X, y)

    def predict(self, X):
        if len(X) == 0:
            return np.array([])
        return self.model.predict(X)


class EnsembleOrchestrator:
    def __init__(
        self,
        data_manager,
        start_train_date: str,
        start_test_date: str,
        end_date: str,
        train_window_days: int = 365 * 3,
        step_days: int = 14,
        horizons: List[int] = [14, 42, 182],
    ):
        self.dm = data_manager

        self.start_train_date = pd.to_datetime(start_train_date)
        self.start_test_date = pd.to_datetime(start_test_date)
        self.end_date = pd.to_datetime(end_date)

        self.train_window_days = train_window_days
        self.step_days = step_days
        self.horizons = horizons

        self.models = {h: XGBModelWrapper() for h in horizons}

        self.pred_registry = []
        self.errors = {h: [] for h in horizons}

    def _compute_weights(self):
        """
        Compute ensemble weights based on historical prediction errors
        """
        weights = {}
        for h in self.horizons:
            errs = self.errors[h]
            if len(errs) == 0:
                weights[h] = 1.0
            else:
                weights[h] = 1.0 / (np.mean(errs) + 1e-6)

        s = sum(weights.values())
        return {k: v / s for k, v in weights.items()}

    def _train_models(self, current_date: pd.Timestamp):
        """
        Train models for all horizons using expanding window
        """
        train_start = current_date - pd.Timedelta(days=self.train_window_days)

        for h in self.horizons:
            X, y = self.dm.get_training_data(
                start_date=str(train_start.date()),
                end_date=str(current_date.date()),
                target_col="target_return",
            )

            self.models[h].fit(X, y)

    def predict(self, df_day: pd.DataFrame):
        """
        Generate ensemble prediction using weighted average of individual models
        """
        if df_day.empty:
            return np.array([])

        X = df_day[self.dm.feature_cols].values

        preds_by_model = {}
        for h in self.horizons:
            preds_by_model[h] = self.models[h].predict(X)

        weights = self._compute_weights()

        ensemble_pred = np.zeros(len(X))
        for h in self.horizons:
            ensemble_pred += weights[h] * preds_by_model[h]

        return ensemble_pred

    def _update_errors(self, current_date: pd.Timestamp):
        """
        Update prediction errors using delayed feedback mechanism
        """
        for record in self.pred_registry:
            pred_date = record["date"]

            if (current_date - pred_date).days < self.dm.horizon_days:
                continue

            df_true_full = self.dm.df[self.dm.df[self.dm.date_col] == current_date]

            if df_true_full.empty:
                continue

            y_true = df_true_full[["SECID", "target_return"]].values
            ids_pred = record["ids"]
            y_pred = record["preds"]

            if len(y_true) != len(y_pred):
                continue

            true_map = {row[0]: row[1] for row in y_true}

            aligned_true = np.array([true_map.get(secid, np.nan) for secid in ids_pred])

            mask = ~np.isnan(aligned_true)

            err = np.mean(np.abs(aligned_true[mask] - y_pred[mask]))

            for h in self.horizons:
                self.errors[h].append(err)

    def run(self):
        """
        Execute main training and prediction loop
        """
        current_date = self.start_test_date

        while current_date <= self.end_date:
            print(f"\n[{current_date.date()}]")

            self._train_models(current_date)
            self._predict(current_date)
            self._update_errors(current_date)

            weights = self._compute_weights()
            print("weights:", weights)
            print("errors:", {k: np.mean(v) if v else None for k, v in self.errors.items()})

            current_date += pd.Timedelta(days=self.step_days)

    def get_last_errors(self) -> Dict[int, float]:
        """
        Returns the last recorded error for each horizon
        """
        last_errors = {}
        
        for h in self.horizons:
            if len(self.errors[h]) > 0:
                last_errors[h] = self.errors[h][-1]
            else:
                last_errors[h] = None
        
        return last_errors


def portfolio_pipeline(
    df_day: pd.DataFrame,
    model,
    feature_cols,
    data_manager,
    secid_col="SECID",
    top_k_small=2,
    top_k_large=3,
):
    """
    Construct portfolio from daily predictions using clustering and top selection
    """
    df = df_day.copy()

    print("rows before filter:", len(df))
    print("rows after filter:", len(df[df["pred"] > 0]))
    df = df[df["pred"] > 0]

    if df.empty:
        return pd.DataFrame()

    clusterer = HDBSCAN(
        min_cluster_size=3,
        min_samples=3,
        metric="euclidean",
        cluster_selection_method="eom"
    )
    numeric_features = [c for c in feature_cols if c != secid_col]
    df["cluster"] = clusterer.fit_predict(df[numeric_features].values)
    if df.empty:
        return pd.DataFrame()

    selected = []
    print("clusters:", np.unique(df["cluster"], return_counts=True))
    for c in df["cluster"].unique():
        cluster_df = df[df["cluster"] == c]
        cluster_df = cluster_df.sort_values("pred", ascending=False)

        k = top_k_large if len(cluster_df) > 5 else top_k_small
        selected.append(cluster_df.head(k))

    selected = pd.concat(selected)

    selected["weight"] = selected["pred"] / selected["pred"].sum()

    selected = selected.copy()
    selected["SECID"] = selected["SECID"].apply(data_manager.id2secid.get)

    return selected[["SECID", "pred", "weight", "cluster"]]


def evaluate_portfolio(portfolio, date, data_manager):
    """
    Calculate realized portfolio return at the horizon date
    """
    horizon = data_manager.horizon_days
    future_date = pd.to_datetime(date) + pd.Timedelta(days=horizon)

    future_date = data_manager._get_nearest_available_date(future_date)

    if future_date is None:
        print("No future data")
        return None

    df_future = data_manager.df[
        data_manager.df[data_manager.date_col] == future_date
    ][["SECID", "target_return"]].copy()

    df_future["SECID"] = df_future["SECID"].map(data_manager.id2secid)

    merged = portfolio.merge(df_future, on="SECID", how="left")

    merged = merged.dropna(subset=["target_return"])

    if merged.empty:
        print("No matched future returns")
        return None

    portfolio_return = (merged["weight"] * merged["target_return"]).sum()

    return portfolio_return, merged


def run_backtest(
    df,
    feature_cols,
    data_manager,
    start_train_date,
    start_test_date,
    end_date,
    portfolio_pipeline_fn,
    save_path="ensemble_180.pkl",
    step_days=14,
    portfolio_dates=None,
):
    """
    Execute full backtest: training loop followed by portfolio construction and evaluation
    """
    orchestrator = EnsembleOrchestrator(
        data_manager=data_manager,
        start_train_date=start_train_date,
        start_test_date=start_test_date,
        end_date=end_date,
        horizons=[182],
        step_days=step_days,
    )

    dates = pd.date_range(start_test_date, end_date, freq=f"{step_days}D")

    for current_date in tqdm(dates, desc="Training"):
        print(f"\nTRAIN DATE: {current_date.date()}")

        orchestrator._train_models(current_date)

        df_day = data_manager.get_features_by_date(current_date)
        if not df_day.empty:
            preds = orchestrator.predict(df_day)

            orchestrator.pred_registry.append({
                "date": current_date,
                "ids": df_day["SECID"].values,
                "preds": preds,
            })

        orchestrator._update_errors(current_date)
    last_errors = {h: errs[-1] if errs else None for h, errs in orchestrator.errors.items()}
    print(f"Last Errors:")
    for h, err in last_errors.items():
        print(f"  Horizon {h}: {err}")
    joblib.dump(orchestrator, save_path)
    print(f"\nMODEL SAVED -> {save_path}")

    portfolios = []

    if portfolio_dates is not None:
        print("\n=== PORTFOLIO TEST ===")

        for d in tqdm(portfolio_dates, desc="Portfolio"):
            d = pd.to_datetime(d)
            print(f"\nPORTFOLIO DATE: {d.date()}")

            df_day = data_manager.get_features_by_date(d)

            if df_day.empty:
                print("No data for this date.")
                continue

            preds = orchestrator.predict(df_day)

            df_day = df_day.copy()
            df_day["pred"] = preds

            portfolio = portfolio_pipeline_fn(
                df_day=df_day,
                model=None,
                feature_cols=feature_cols,
                data_manager=data_manager,
            )
            print(f"Selected {len(portfolio)} stocks for portfolio.")

            if not portfolio.empty:
                portfolios.append({
                    "date": d,
                    "portfolio": portfolio
                })

                print(portfolio.head())
                result = evaluate_portfolio(portfolio, d, data_manager)

                if result is not None:
                    port_ret, details = result

                    print(portfolio.head())
                    print("\nPortfolio return:", round(port_ret, 4))
                    print("Details:")
                    print(details.head())

                    print(portfolios)


if __name__ == "__main__":
    df = pd.read_csv('output_with_cbr_rate.csv', parse_dates=['TRADEDATE'])

    feature_cols = [
        "momentum_6m", "momentum_12m", "momentum_24m", "momentum_36m",
        "volatility_6m", "volatility_12m", "volatility_24m", "volatility_36m",
        "max_drawdown_6m", "max_drawdown_12m", "max_drawdown_24m", "max_drawdown_36m",
        "avg_value_6m", "avg_value_12m", "avg_value_36m",
        "dividend_yield_12m", "dividend_yield_3y_avg", "dividend_consistency_3y",
        "relative_strength_12m", "excess_return_6m", "beta_12m",
        "log_issuesize", "cbr_key_rate"
    ]

    dm = DataManager(
        df=df,
        feature_cols=feature_cols,
        delay_cols=["target_return"],
        date_col="TRADEDATE",
        id_col="SECID",
        use_scaler=True,
        horizon_days=180,
    )

    dm.fit_scaler(start_date="2015-01-01", end_date="2018-01-01")
    dm.apply_scaler()

    history = run_backtest(
        df=df,
        feature_cols=feature_cols,
        data_manager=dm,
        start_train_date="2015-01-01",
        start_test_date="2018-01-01",
        end_date="2024-01-01",
        portfolio_pipeline_fn=portfolio_pipeline,
        portfolio_dates=[
            "2024-08-05",
            "2024-09-03",
            "2025-04-03",
            "2025-05-05",
            "2025-06-06"
        ]
    )

Training:   0%|          | 0/157 [00:00<?, ?it/s]


TRAIN DATE: 2018-01-01


Training:   1%|          | 1/157 [00:00<01:06,  2.33it/s]


TRAIN DATE: 2018-01-15


Training:   1%|▏         | 2/157 [00:00<00:55,  2.81it/s]

...

Last Errors:
  Horizon 182: 0.8379667682130081

MODEL SAVED -> ensemble_180.pkl

=== PORTFOLIO TEST ===


Portfolio:   0%|          | 0/5 [00:00<?, ?it/s]


PORTFOLIO DATE: 2024-08-05
rows before filter: 132
rows after filter: 72
clusters: (array([-1,  0,  1,  2]), array([26,  4,  3, 39]))
Selected 10 stocks for portfolio.
        SECID      pred    weight  cluster
208301   DZRD  0.510278  0.137263       -1
208358   NKHP  0.490775  0.132017       -1
208378   RTSB  0.410715  0.110481       -1
208339   MISB  0.679111  0.182678        2
208340  MISBP  0.498367  0.134059        2
        SECID      pred    weight  cluster
208301   DZRD  0.510278  0.137263       -1
208358   NKHP  0.490775  0.132017       -1
208378   RTSB  0.410715  0.110481       -1
208339   MISB  0.679111  0.182678        2
208340  MISBP  0.498367  0.134059        2

Portfolio return: -0.1812
Details:
   SECID      pred    weight  cluster  target_return
0   DZRD  0.510278  0.137263       -1      -0.246440
1   NKHP  0.490775  0.132017       -1      -0.187500
2   RTSB  0.410715  0.110481       -1      -0.360902
3   MISB  0.679111  0.182678        2      -0.206400
4  MISBP  0.49

Portfolio: 100%|██████████| 5/5 [00:00<00:00, 41.50it/s]

rows before filter: 130
rows after filter: 73
clusters: (array([-1,  0,  1,  2]), array([42,  7, 15,  9]))
Selected 12 stocks for portfolio.
        SECID      pred    weight  cluster
233334   DZRD  1.221825  0.175895       -1
233397  OMZZP  1.026411  0.147763       -1
233319   ASSB  0.716363  0.103128       -1
233324  BISVP  0.394564  0.056802        1
233356  KRSBP  0.314208  0.045234        1
        SECID      pred    weight  cluster
233334   DZRD  1.221825  0.175895       -1
233397  OMZZP  1.026411  0.147763       -1
233319   ASSB  0.716363  0.103128       -1
233324  BISVP  0.394564  0.056802        1
233356  KRSBP  0.314208  0.045234        1

Portfolio return: 0.2386
Details:
   SECID      pred    weight  cluster  target_return
2   ASSB  0.716363  0.103128       -1       0.932773
3  BISVP  0.394564  0.056802        1       0.116356
4  KRSBP  0.314208  0.045234        1       0.268959
5   KRSB  0.266784  0.038406        1       0.235354
6   YAKG  0.267653  0.038532        0      